In [1]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.merge import merge
from pathlib import Path
import glob, time

# ===============================
# 参数区
# ===============================
YEARS = range(2012, 2023)
TILES = ["N50_30", "N51_30"]
TARGET_CRS = "EPSG:4547"
CROPLAND_CODES = {1}
FALLOW_THR = 0.30
FALLOW_NODATA = 255

# ===============================
# 主循环
# ===============================
for year in YEARS:
    print(f"\n==============================")
    print(f"🌾 Processing year {year}")
    print(f"==============================")
    t0 = time.time()
    
    # === 路径设置 ===
    fvc_dir = Path(f"/Users/wangze/cropland_data/fvc/{year}")
    fvc_dev = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc")
    fvc_dev.mkdir(exist_ok=True, parents=True)
    
    clcd_src = Path(f"/Users/wangze/Dropbox/Emi/LandControl/geodata/CLCD_v01_{year}_albert_province/CLCD_v01_{year}_albert_jiangsu.tif")
    clcd_out = Path(f"/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/CLCD_v01_{year}_albert_jiangsu_albers.tif")
    
    # ------------------------------
    # (1) FVC tiles: 计算平均 + 重投影
    # ------------------------------
    for tile in TILES:
        print(f"\n🌍 Tile {tile}: averaging 24 half-month files ...")
        files = sorted(glob.glob(str(fvc_dir / f"{tile}_FVC-{year}-*-30m.tif")))
        if not files:
            print(f"⚠️ No files found for {tile}")
            continue
        
        with rasterio.open(files[0]) as ref:
            ref_meta = ref.meta.copy()
            shape = (ref.height, ref.width)
            transform = ref.transform
        
        mean_fvc = np.zeros(shape, dtype=np.float64)
        count = np.zeros(shape, dtype=np.int32)
        
        for i, f in enumerate(files, 1):
            with rasterio.open(f) as src:
                arr = src.read(1).astype(np.float32)
                arr[arr > 100] = np.nan
                arr /= 100.0
                mask = np.isfinite(arr)
                mean_fvc[mask] += arr[mask]
                count[mask] += 1
            print(f"  ✅ {i:02d}/24 processed")
        
        mean_fvc = np.divide(mean_fvc, count, out=np.zeros_like(mean_fvc), where=count>0).astype(np.float32)
        
        raw_path = fvc_dev / f"FVC_{year}_{tile}_mean_raw.tif"
        meta = ref_meta.copy()
        meta.update(driver="GTiff", dtype="float32", count=1)
        with rasterio.open(raw_path, "w", **meta) as dst:
            dst.write(mean_fvc, 1)
        
        reproj_path = fvc_dev / f"FVC_{year}_{tile}_mean_4547.tif"
        with rasterio.open(raw_path) as src:
            transform, w, h = calculate_default_transform(src.crs, TARGET_CRS, src.width, src.height, *src.bounds)
            kwargs = src.meta.copy()
            kwargs.update(crs=TARGET_CRS, transform=transform, width=w, height=h, dtype="float32")
            with rasterio.open(reproj_path, "w", **kwargs) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.bilinear)
        print(f"  🌐 Reprojected → {reproj_path}")

    # ------------------------------
    # (2) CLCD 重投影到 EPSG:4547
    # ------------------------------
    print(f"\n🗺 Reprojecting CLCD for {year} ...")
    with rasterio.open(clcd_src) as src:
        transform, w, h = calculate_default_transform(src.crs, TARGET_CRS, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update(crs=TARGET_CRS, transform=transform, width=w, height=h)
        with rasterio.open(clcd_out, "w", **kwargs) as dst:
            for i in range(1, src.count+1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.nearest)
    print(f"  ✅ CLCD reprojected: {clcd_out.name}")

    # ------------------------------
    # (3) Merge N50+N51
    # ------------------------------
    print("\n🔗 Merging FVC tiles ...")
    fvc50 = fvc_dev / f"FVC_{year}_N50_30_mean_4547.tif"
    fvc51 = fvc_dev / f"FVC_{year}_N51_30_mean_4547.tif"
    srcs = [rasterio.open(fvc50), rasterio.open(fvc51)]
    mosaic, out_trans = merge(srcs, method="max")
    out_meta = srcs[0].meta.copy()
    out_meta.update(driver="GTiff", height=mosaic.shape[1], width=mosaic.shape[2], transform=out_trans, crs=TARGET_CRS, dtype="float32")
    merged_path = fvc_dev / f"FVC_{year}_Jiangsu_merged_full_4547.tif"
    with rasterio.open(merged_path, "w", **out_meta) as dst:
        dst.write(mosaic[0], 1)
    print(f"  ✅ Merged FVC saved → {merged_path}")

    # ------------------------------
    # (4) 生成撂荒图
    # ------------------------------
    print("\n🌱 Generating fallow map ...")
    with rasterio.open(clcd_out) as clcd_src:
        clcd = clcd_src.read(1)
        clcd_meta = clcd_src.meta.copy()
        clcd_crs = clcd_src.crs
        clcd_tf = clcd_src.transform
        clcd_shape = (clcd_src.height, clcd_src.width)

    with rasterio.open(merged_path) as fvc_src:
        fvc = fvc_src.read(1).astype(np.float32)
        fvc_crs = fvc_src.crs
        fvc_tf = fvc_src.transform

    if fvc_crs != clcd_crs or fvc.shape != clcd_shape or fvc_tf != clcd_tf:
        aligned_fvc = np.full(clcd_shape, np.nan, dtype=np.float32)
        reproject(
            source=fvc,
            destination=aligned_fvc,
            src_transform=fvc_tf,
            src_crs=fvc_crs,
            dst_transform=clcd_tf,
            dst_crs=clcd_crs,
            resampling=Resampling.bilinear,
            src_nodata=None,
            dst_nodata=np.nan)
        fvc = aligned_fvc
        print("  ✅ FVC aligned to CLCD grid")

    fvc[(fvc < 0) | (fvc > 1)] = np.nan
    cropland_mask = np.isin(clcd, list(CROPLAND_CODES))
    valid_on_cropland = cropland_mask & np.isfinite(fvc)
    fallow_bool = np.zeros_like(fvc, dtype=bool)
    fallow_bool[valid_on_cropland] = (fvc[valid_on_cropland] < FALLOW_THR)
    fallow_u8 = np.full(fvc.shape, FALLOW_NODATA, dtype=np.uint8)
    fallow_u8[valid_on_cropland] = fallow_bool[valid_on_cropland].astype(np.uint8)
    
    total_cropland = int(cropland_mask.sum())
    valid_cropland = int(valid_on_cropland.sum())
    fallow_pixels = int(fallow_bool.sum())
    rate = fallow_pixels / valid_cropland if valid_cropland > 0 else np.nan
    print(f"  📊 Fallow rate ({year}): {rate:.2%}")
    
    out_fvc_tif = fvc_dev / f"FVC_Jiangsu_{year}_cropland_only_4547.tif"
    fvc_meta = clcd_meta.copy()
    fvc_meta.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan)
    with rasterio.open(out_fvc_tif, "w", **fvc_meta) as dst:
        dst.write(fvc.astype(np.float32), 1)
    print(f"  ✅ Cropland FVC saved: {out_fvc_tif.name}")

    print(f"⏱ Year {year} completed in {(time.time()-t0)/60:.1f} min")

print("\n🎯 All years processed successfully!")



🌾 Processing year 2012

🌍 Tile N50_30: averaging 24 half-month files ...
  ✅ 01/24 processed
  ✅ 02/24 processed
  ✅ 03/24 processed
  ✅ 04/24 processed
  ✅ 05/24 processed
  ✅ 06/24 processed
  ✅ 07/24 processed
  ✅ 08/24 processed
  ✅ 09/24 processed
  ✅ 10/24 processed
  ✅ 11/24 processed
  ✅ 12/24 processed
  ✅ 13/24 processed
  ✅ 14/24 processed
  ✅ 15/24 processed
  ✅ 16/24 processed
  ✅ 17/24 processed
  ✅ 18/24 processed
  ✅ 19/24 processed
  ✅ 20/24 processed
  ✅ 21/24 processed
  ✅ 22/24 processed
  ✅ 23/24 processed
  ✅ 24/24 processed
  🌐 Reprojected → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_fvc/FVC_2012_N50_30_mean_4547.tif

🌍 Tile N51_30: averaging 24 half-month files ...
  ✅ 01/24 processed
  ✅ 02/24 processed
  ✅ 03/24 processed
  ✅ 04/24 processed
  ✅ 05/24 processed
  ✅ 06/24 processed
  ✅ 07/24 processed
  ✅ 08/24 processed
  ✅ 09/24 processed
  ✅ 10/24 processed
  ✅ 11/24 processed
  ✅ 12/24 processed
  ✅ 13/24 processed
  ✅ 14/24 processed
  ✅ 15/24 